#####   行为数据预处理 Pipeline：将实验硬件event文件（.evt）解析并修复后输出标准化的 .pkl 文件

内核：base

In [1]:
dir = '/mnt/HDD1/zyyk78/Stage4/'  # 可设为任意父目录（甚至 Stage4 这一层），脚本会逐个进入每个鼠的子目录、单独跑 README 匹配
#home/zyyk78/HDD/zyyk78/（软连接快捷方式）
log_dir='/mnt/HDD1/zyyk78/llom_log'

准备好待处理文件夹后，直接运行下列代码即可


In [2]:
#初始化
import os
import fixing
from ParseEvent import ParseEvent
import pickle
import pandas as pd
import warnings
import re
import traceback
warnings.simplefilter(action='ignore', category=FutureWarning)

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

def find_evt_files_without_pkl(parent_dir):
    """
    递归扫描 parent_dir，按子目录分组返回【尚无 pkl 的 evt 文件】。
    返回 dict: {subdir_path: [evt_path, ...], ...}
    下游会逐个子目录加载它自己的 README.xlsx。
    """
    subdir_to_evts = {}
    for root, dirs, files in os.walk(parent_dir):
        evts_in_subdir = []
        for file in files:
            if file.endswith('.evt'):
                pkl_file_name = os.path.splitext(file)[0] + '.pkl'
                if pkl_file_name not in files:
                    evts_in_subdir.append(os.path.join(root, file))
        if evts_in_subdir:
            subdir_to_evts[root] = evts_in_subdir
    return subdir_to_evts

# 收集所有含未处理 evt 的子目录（按子目录分组）
subdir_evt_map = find_evt_files_without_pkl(dir)

total_files = sum(len(v) for v in subdir_evt_map.values())
if total_files == 0:
    print('No evt to be processed!\n')
else:
    print(f'Find {total_files} File(s) across {len(subdir_evt_map)} subdirectory(ies)!')
    for subdir, evts in subdir_evt_map.items():
        print(f'  - {subdir}: {len(evts)} file(s)')

delta1=25000
delta2=10000

Find 78 File(s) across 5 subdirectory(ies)!
  - /mnt/HDD1/zyyk78/Stage4/C05: 4 file(s)
  - /mnt/HDD1/zyyk78/Stage4/C06: 40 file(s)
  - /mnt/HDD1/zyyk78/Stage4/D05: 4 file(s)
  - /mnt/HDD1/zyyk78/Stage4/D10: 27 file(s)
  - /mnt/HDD1/zyyk78/Stage4/A08: 3 file(s)


Process All File Automatically！

In [3]:
# ===== README 数据体检（先跑这个，再跑下面的批量处理）=====
# 扫描 dir 下所有子目录, 把 README.xlsx 的问题一次性列出来, 便于手工补录数据:
#   【2】缺失记录: 有 evt 但 README 里查不到日期        -> 需要补录
#   【3】多余记录: README 有记录但没有 evt / 日期疑似写错  -> 需要核对
#   【4】小数日期: 第1列日期带小数(会被截断取整)          -> 建议改成整数
# 结果直接打印在本格下方, 不导出文件
# 注意: 已有 pkl 的会话不会被重复处理(下一格只处理没有 pkl 的 evt);
#       但缺 README 记录时生成的 pkl 会缺 tMod(模式)字段, 补完数据后需要删除它们才会重跑:
#       readme_check.delete_bad_pkls(readme_report, dry_run=False)
import readme_check
import importlib
importlib.reload(readme_check)          # 改过 readme_check.py 后无需重启内核
readme_report = readme_check.check(dir)
readme_check.show(readme_report)
print('\n补全 README 后, 若要重跑那些缺 tMod 的旧 pkl, 执行:')
print("  readme_check.delete_bad_pkls(readme_report, dry_run=True)   # 先看清单")
print("  readme_check.delete_bad_pkls(readme_report, dry_run=False)  # 确认后再删")



README 数据体检报告

【1】各子目录概况
子目录                          路径  evt总数  待处理(无pkl)  缺README  无README文件
D04 /mnt/HDD1/zyyk78/Stage4/D04     39          0        0      False
D08 /mnt/HDD1/zyyk78/Stage4/D08     27          0        0      False
A07 /mnt/HDD1/zyyk78/Stage4/A07     26          0        0      False
C05 /mnt/HDD1/zyyk78/Stage4/C05     86          4        1      False
C02 /mnt/HDD1/zyyk78/Stage4/C02     42          0        0      False
C07 /mnt/HDD1/zyyk78/Stage4/C07      5          0        0      False
D06 /mnt/HDD1/zyyk78/Stage4/D06     37          0        0      False
D09 /mnt/HDD1/zyyk78/Stage4/D09      7          0        0      False
C03 /mnt/HDD1/zyyk78/Stage4/C03     11          0        0      False
E01 /mnt/HDD1/zyyk78/Stage4/E01     22          0        0      False
C06 /mnt/HDD1/zyyk78/Stage4/C06     78         40        0      False
D05 /mnt/HDD1/zyyk78/Stage4/D05     52          4        0      False
D10 /mnt/HDD1/zyyk78/Stage4/D10     27         27        1      F

In [4]:
# Process All File Automatically!（按子目录分组，每个子目录单独加载 README 匹配）
for subdir, evt_paths in subdir_evt_map.items():
    # 加载当前子目录自己的 README.xlsx
    readme_path = os.path.join(subdir, "README.xlsx")
    df = pd.DataFrame()
    if os.path.exists(readme_path):
        df = pd.read_excel(readme_path, sheet_name='Sheet1', header=None).iloc[2:, :]
        num = pd.to_numeric(df[0], errors='coerce')
        # pandas 3.x 不再允许 float->Int64 的非等价转换: 空值(NaN)没问题, 但带小数的日期(如 20260615.3333, 录表时带进来的
        # Excel 时间分数)会抛 'cannot safely cast non-equivalent float64 to int64'。这里先向下取整(截断, 不能用四舍五入,
        # 否则 20260616.67 会被错变成 20260617), NaN 会自动变成 <NA>。
        df[0] = (num // 1).astype('Int64')
        print(f"\n[{subdir}] README loaded: {len(df)} rows")
    else:
        print(f"\n[{subdir}] No README.xlsx found, README matching will be skipped for this subdir.")

    sub_count = 0
    while sub_count < len(evt_paths):
        evt_path = evt_paths[sub_count]
        print(f'Processing file {sub_count+1}/{len(evt_paths)} in {subdir}: {evt_path}')
        try:
            Event = ParseEvent(evt_path)
        except Exception as e:
            print('An error occoure while processing Event!' , e)
            Event = {
                'SlnL':{ 'Time_S':[],'Time_E':[]},
                'SlnR':{ 'Time_S':[],'Time_E':[]},
                'ROmL':{ 'Time_S':[],'Time_E':[]},
                'ROmR':{ 'Time_S':[],'Time_E':[]},
                'ModL':{ 'Time_S':[],'Time_E':[]},
                'ModR':{ 'Time_S':[],'Time_E':[]},
                'LickL':{ 'Time_S':[],'Time_E':[]},
                'LickR':{ 'Time_S':[],'Time_E':[]}
            }
        pattern = r'\d{8}'
        target_date = re.findall(pattern, os.path.basename(evt_path).split('_')[0])[0]
        log_file = os.path.join(log_dir, f"log{target_date}.txt")
        matched_row = pd.DataFrame()

        if not df.empty:
            matched_row = df[df[0]==int(target_date)]
            if len(matched_row) > 0:
                print('Find matched row in CSV data!')
                print(f'Target Tring mode: {matched_row.iloc[0,2]}')
            else:
                print(f'⚠️ WARNING: No matching row in README.xlsx for date {target_date}. Please add the record to README!')
        try:
            Event = fixing.fixing(Event,26000,39000,log_file,matched_row)
        except Exception as e:
            print(f'Error occurred while fixing the event: {e}')
            traceback.print_exc()
            Event = None
        if (Event != None):
            pkl_path = os.path.splitext(evt_path)[0] + '.pkl'
            with open(pkl_path, "wb") as pkl_f:
                pickle.dump(Event, pkl_f)
                Event = None
            print('File Saved to: ' + pkl_path)
        else:
            print('Event = Null!! Pkl won\'t be saved!')
        sub_count += 1

print('\nAll Finished!\n')


[/mnt/HDD1/zyyk78/Stage4/C05] README loaded: 90 rows
Processing file 1/4 in /mnt/HDD1/zyyk78/Stage4/C05: /mnt/HDD1/zyyk78/Stage4/C05/20261016_F9W_00.evt
Find matched row in CSV data!
Target Tring mode: F9W
Removed an abnormal event
Log file /mnt/HDD1/zyyk78/llom_log/log20261016.txt found. Event is matched to 2, time difference is 47 seconds.
No Reference Event! Cannot fix! 
No Reference Event! Cannot fix! 
Fixing with log! 
Matching SlnL
27 events in hw, 27 events in sw, 27 matched!
Fit data: a=1.00, b=0.61(s), r^2=1.00
ROmL Event too short! Removed!
1 Events matched! a=1.00±0.00, b=0.61±0.00(s)
No SlnR Event! Fixing 
No ROmR Event! Fixing 
File Saved to: /mnt/HDD1/zyyk78/Stage4/C05/20261016_F9W_00.pkl
Processing file 2/4 in /mnt/HDD1/zyyk78/Stage4/C05: /mnt/HDD1/zyyk78/Stage4/C05/20261019_F9W_00.evt
Find matched row in CSV data!
Target Tring mode: F9W
Removed an abnormal event
Log file /mnt/HDD1/zyyk78/llom_log/log20261019.txt found. Event is matched to 3, time difference is 0 second

/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


⚠️ WARNING: No matching row in README.xlsx for date 20261012. Please add the record to README!
Removed an abnormal event
Removed an abnormal event
Removed an abnormal event
Removed an abnormal event
Removed an abnormal event
Error occurred while fixing the event: invalid literal for int() with base 10: ''
Event = Null!! Pkl won't be saved!
Processing file 4/4 in /mnt/HDD1/zyyk78/Stage4/C05: /mnt/HDD1/zyyk78/Stage4/C05/20260714_F9F_00.evt


Traceback (most recent call last):
  File "/tmp/ipykernel_236957/4261776754.py", line 48, in <module>
    Event = fixing.fixing(Event,26000,39000,log_file,matched_row)
  File "/home/jyy/Documents/Beh_Ana_J/fixing.py", line 536, in fixing
    event_num=int(input(f"Please input the event number of event: ")) - 1
              ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: ''


Find matched row in CSV data!
Target Tring mode: F9F
Removed an abnormal event
Removed an abnormal event
Removed an abnormal event
Removed an abnormal event
Log file /mnt/HDD1/zyyk78/llom_log/log20260714.txt found. Event is matched to 0, time difference is 1 seconds.
No ROmL Event! Fixing 
No ModR Event! Fixing 
Fixing with log! 
SlnL Event too short! Removed!
Matching SlnR
57 events in hw, 57 events in sw, 57 matched!
Fit data: a=1.00, b=-3.12(s), r^2=1.00
Matching ROmR
160 events in hw, 160 events in sw, 160 matched!
Fit data: a=1.00, b=-3.12(s), r^2=1.00
2 Events matched! a=1.00±0.00, b=-3.12±0.00(s)
No ROmL Event! Fixing 
No ModR Event! Fixing 
File Saved to: /mnt/HDD1/zyyk78/Stage4/C05/20260714_F9F_00.pkl

[/mnt/HDD1/zyyk78/Stage4/C06] README loaded: 84 rows
Processing file 1/40 in /mnt/HDD1/zyyk78/Stage4/C06: /mnt/HDD1/zyyk78/Stage4/C06/20260923_T9W_00.evt
Find matched row in CSV data!
Target Tring mode: T9W
Removed an abnormal event
Removed an abnormal event
Too long event detec

...

Find matched row in CSV data!
Target Tring mode: P9F
Removed an abnormal event
Removed an abnormal event
Removed an abnormal event
Log file /mnt/HDD1/zyyk78/llom_log/log20260913.txt found. Event is matched to 3, time difference is 0 seconds.
No ROmL Event! Fixing 
No Mod info!
Fixing with log! 
SlnL Event too short! Removed!
SlnR Event too short! Removed!
ROmR Event too short! Removed!
0 Events matched! a=nan±nan, b=nan±nan(s)
No ROmL Event! Fixing 
No ModL Event! Fixing 
No ModR Event! Fixing 
File Saved to: /mnt/HDD1/zyyk78/Stage4/D10/20260913_P9F_00.pkl
Processing file 13/27 in /mnt/HDD1/zyyk78/Stage4/D10: /mnt/HDD1/zyyk78/Stage4/D10/20261021_Alm25P7F_00.evt


/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Find matched row in CSV data!
Target Tring mode: Alm25P7F
Log file /mnt/HDD1/zyyk78/llom_log/log20261021.txt found. Event is matched to 1, time difference is 0 seconds.
No ROmL Event! Fixing 
No Mod info!
Fixing with log! 
SlnL Event too short! Removed!
Matching SlnR
105 events in hw, 348 events in sw, 105 matched!
Fit data: a=1.00, b=-2379.28(s), r^2=1.00
Matching ROmR
279 events in hw, 828 events in sw, 279 matched!
Fit data: a=1.00, b=-2382.05(s), r^2=1.00
2 Events matched! a=1.00±0.00, b=-2380.67±1.39(s)
No ROmL Event! Fixing 
No ModL Event! Fixing 
No ModR Event! Fixing 
File Saved to: /mnt/HDD1/zyyk78/Stage4/D10/20261021_Alm25P7F_00.pkl
Processing file 14/27 in /mnt/HDD1/zyyk78/Stage4/D10: /mnt/HDD1/zyyk78/Stage4/D10/20261008_T7F_00.evt
Find matched row in CSV data!
Target Tring mode: T7F
Removed an abnormal event
Too long event detected![(2487621, 2570703), (62172941, 62355242)]
No missing event, skipping fixing
File Saved to: /mnt/HDD1/zyyk78/Stage4/D10/20261008_T7F_00.pkl
Proc

/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/jyy/.local/lib/python3.14/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Find matched row in CSV data!
Target Tring mode: T9F
No missing event, skipping fixing
File Saved to: /mnt/HDD1/zyyk78/Stage4/D10/20260922_T9F_00.pkl
Processing file 17/27 in /mnt/HDD1/zyyk78/Stage4/D10: /mnt/HDD1/zyyk78/Stage4/D10/20261018_Arm25P7F_00.evt
Find matched row in CSV data!
Target Tring mode: Arm25P7F
Too long event detected![(4543209, 4596336), (16757718, 16816810), (16900409, 16971000), (18357666, 18413452), (18509377, 18561075), (18598903, 18658573), (21022543, 21084089), (34973473, 35056527), (35107405, 35180073), (38101555, 38159748), (38531891, 38606627), (38690863, 38755683), (39936576, 40001934), (40146645, 40228902), (40994501, 41052375), (45443124, 45551460), (52929178, 53014667), (53323825, 53381689), (56270417, 56323607), (56412529, 56467152), (59564748, 59623946), (62365473, 62427151), (66024517, 66092924), (66112818, 66295231), (82396297, 82462838), (82478870, 82669622), (82751556, 82832064), (82834309, 82908823), (82917908, 82984999), (83730874, 83787674), (8

#####   补充功能一：补全缺失起止时间的Readme
根据evt文件的修改时间，匹配对应llom_log日志中的起止时间

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import re

# 输入指定目录，以及目标日期区间
# -------------------------------------------------------------------------------

path = Path("/mnt/HDD1/zyyk78/MPtest/B05")
start_date = datetime.strptime("20250725", "%Y%m%d")
end_date = datetime.strptime("20250727", "%Y%m%d")

# -------------------------------------------------------------------------------

log_path = Path("/mnt/HDD1/zyyk78/llom_log")
# 获取所有 EVT 文件及其修改时间
evt_files = list(path.rglob("*.evt"))
files_with_mtime = []
for file in evt_files:
    if len(file.name) >= 11 and file.name[:8].isdigit():
        date_str = file.name[:8]
        try:
            file_date = datetime.strptime(date_str, "%Y%m%d")
            if start_date <= file_date <= end_date:
                stat_info = file.stat()
                mtime = datetime.fromtimestamp(stat_info.st_mtime)
                files_with_mtime.append((mtime, file))
        except ValueError:
            pass

files_with_mtime.sort(key=lambda x: x[0])

print(f"当前指定目录为: {path}")

for mtime, evt_file in files_with_mtime:
    # 获取日期对应的 log 文件
    date_str = evt_file.name[:8]
    log_file = log_path / f"log{date_str}.txt"
    
    if not log_file.exists():
        print(f"No log file for {date_str}")
        continue
    
    # 解析 log 文件中的所有 start 和 r 行
    log_lines = []
    with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line_lower = line.lower()
            if '->start' in line_lower or '->trial start' in line_lower or '<-r' in line_lower:
                match = re.match(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\.\d+', line)
                if match:
                    log_time = datetime.strptime(match.group(1), "%Y-%m-%d %H:%M:%S")
                    log_lines.append((log_time, line.rstrip()))
    
    # 找到最接近 EVT mtime 的 r 行（在10分钟误差内）
    target_time = mtime.replace(second=mtime.second, microsecond=0)
    time_diff = timedelta(minutes=10)
    matched_r = None
    
    for log_time, log_line in log_lines:
        if '<-r' in log_line.lower():
            diff = abs((log_time - target_time).total_seconds())
            if diff < time_diff.total_seconds():
                time_diff = timedelta(seconds=diff)
                matched_r = (log_time, log_line)
    
    if matched_r:
        log_time, r_line = matched_r
        # 找到 r 行的索引
        r_index = None
        for i, (lt, ll) in enumerate(log_lines):
            if lt == log_time and '<-r' in ll.lower():
                r_index = i
                break
        
        if r_index is not None:
            # 找到上一个 r 的位置
            prev_r_index = None
            for i in range(r_index - 1, -1, -1):
                if '<-r' in log_lines[i][1].lower():
                    prev_r_index = i
                    break
            
            start_idx = prev_r_index + 1 if prev_r_index is not None else 0
            
            # 获取开始时间和结束时间
            start_time = log_lines[start_idx][0]
            end_time = log_lines[r_index][0]
            duration = end_time - start_time
            
            # 格式化输出
            evt_name = evt_file.name.replace('.evt', '')
            start_str = start_time.strftime('%H:%M:%S')
            end_str = end_time.strftime('%H:%M:%S')
            duration_str = f"{int(duration.total_seconds()//3600):02d}:{int((duration.total_seconds()%3600)//60):02d}:{int(duration.total_seconds()%60):02d}"
            
            print(f"{evt_name}，开始时间：{start_str}，结束时间：{end_str}，用时{duration_str}")

#####   觉得有不确定的地方，可以手动检查evt和txt日志文件：


1. 检索指定目录下、指定时间区间的evt文件及其文件修改时间(跑了就懂)

In [ ]:
from pathlib import Path
from datetime import datetime

# -------------------------------------------------------------------------------
# path = "/home/zyyk78/HDD/zyyk78/Stage4"
# path = Path(path)
pkl_files = list(path.rglob("*.evt"))

# start = datetime.strptime("20260414", "%Y%m%d")
# end = datetime.strptime("20260414", "%Y%m%d")
# -------------------------------------------------------------------------------

files_with_mtime = []
for file in pkl_files:
    if len(file.name) >= 11 and file.name[:8].isdigit():
        date_str = file.name[:8]
        try:
            file_date = datetime.strptime(date_str, "%Y%m%d")
            if start <= file_date <= end:
                # Get file stats
                stat_info = file.stat()
                ctime = datetime.fromtimestamp(stat_info.st_ctime)
                mtime = datetime.fromtimestamp(stat_info.st_mtime)
                files_with_mtime.append((mtime, ctime, stat_info.st_size, file))
        except ValueError:
            pass

# Sort by modified time
files_with_mtime.sort(key=lambda x: x[0])

# Print in sorted order
for mtime, ctime, size, file in files_with_mtime:
    print(f"File: {file}")
    # print(f"  Created:  {ctime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Size: {size} bytes")
    print()

2.输出当天日志出现过的start和reset

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

# -------------------------------------------------------------------------------

path = Path("/home/zyyk78/HDD/zyyk78/llom_log")
txt_files = list(path.rglob("*.txt"))

# start = datetime.strptime("20260414", "%Y%m%d")
# end = datetime.strptime("20260414", "%Y%m%d")

# -------------------------------------------------------------------------------

existing_files = {}
for file in txt_files:
    if file.name.startswith("log") and len(file.name) >= 11:
        date_str = file.name[3:11]
        try:
            file_date = datetime.strptime(date_str, "%Y%m%d")
            existing_files[file_date] = file
        except ValueError:
            pass

current = start
while current <= end:
    if current not in existing_files:
        current += timedelta(days=1)
        continue
    
    file = existing_files[current]
    print(f"\n=== {file} ===")
    skip_reset = False
    with open(file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line_lower = line.lower()
            if 'start washing' in line_lower:
                skip_reset = True
            elif ('->start' in line_lower or "->trial start" in line_lower) and 'start washing' not in line_lower:
                skip_reset = False
            if ('->start' in line_lower or "->trial start" in line_lower) or '<-r' in line_lower:
                if not skip_reset:
                    print(line.rstrip())
    
    current += timedelta(days=1)